# Integrity-aware adaptive fusion for plant disease recognition

Same audit (64-bit dHash, multi-index radius 5, thumbnail correlation >= 0.90) and the same fixed clean test sets as the
clean-and-balance grid. Models: fine-tuned EfficientNet-B0; frozen DINOv2-L and CLIP ViT-L with a linear + prototype
head; fusion of DINOv2-L + CLIP ViT-L (fusion2) and of EfficientNet + DINOv2-L + CLIP ViT-L (fusion3). Adaptive model: validation-gated weighting of the CNN branch against the foundation-model fusion, temperature scaling on the same validation split (10% of training, group-aware). Training labels:
keep (raw), drop (conflicting groups removed, one image per duplicate group), soft (conflict-aware soft labels, one unit
of weight per duplicate group). Kaggle: GPU T4, Internet ON, inputs: paddy-disease-classification and
cassava-leaf-disease-classification competitions.

In [ ]:
RUN = ["plantdoc", "paddydoctor"]          # edit: any of "plantdoc", "paddydoctor", "cassava2020", "plantwild", "rice_leaf"
VIT_ON = ["plantdoc"]                      # datasets on which ViT-S also runs the full grid
GRID_ON = None                             # datasets that run the clean/balance grid (None = all of RUN)
LEAK_ON = []                               # datasets that also run the leakage test (naive random split vs copy-free test images)
import os, sys, json, time, glob, subprocess, random, io, hashlib
from collections import defaultdict
from pathlib import Path
import numpy as np
T0 = time.time()
def log(m): print("[%7.1fs] %s" % (time.time() - T0, m), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import timm
DEV = "cuda" if torch.cuda.is_available() else "cpu"
log("device %s %s" % (DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else ""))
IMG, CACHE_PX, EPOCHS, BATCH, LR, LR_VIT = 224, 256, 15, 64, 1e-3, 1e-4
SEEDS = [42, 43, 44]
RADIUS, VERIFY = 5, 0.90
SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE: EPOCHS, SEEDS = 1, [42]
if os.environ.get("RUN"): RUN = os.environ["RUN"].split(",")
if os.environ.get("GRID_ON") is not None: GRID_ON = [x for x in os.environ["GRID_ON"].split(",") if x]
if os.environ.get("LEAK_ON") is not None: LEAK_ON = [x for x in os.environ["LEAK_ON"].split(",") if x]
if GRID_ON is None: GRID_ON = list(RUN)
TMP = Path(os.environ.get("TMP_DIR", "/tmp/benchdata")); TMP.mkdir(parents=True, exist_ok=True)   # datasets live here, not in the output folder
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working")); OUTD = WORK / "integrity_out"; OUTD.mkdir(parents=True, exist_ok=True)
WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 2)
RESULTS = {"_settings": {"img": IMG, "epochs": EPOCHS, "batch": BATCH, "lr": LR, "lr_vit": LR_VIT, "seeds": SEEDS,
                         "radius": RADIUS, "verify_corr": VERIFY, "gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu"}}
def save(): (OUTD / "integrity_results.json").write_text(json.dumps(RESULTS, indent=1))
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# loading: every image is decoded once, resized to 256 x 256 and kept in memory with its signature
def decode(path):
    """path (or raw bytes) -> (256x256x3 uint8, dHash, 32x32 grey thumbnail, md5) or None if unreadable."""
    try:
        data = path if isinstance(path, bytes) else open(path, "rb").read()
        with Image.open(io.BytesIO(data)) as im:
            im.draft("RGB", (CACHE_PX * 2, CACHE_PX * 2))
            rgb = im.convert("RGB"); g = rgb.convert("L")
            a = np.asarray(g.resize((9, 8), Image.LANCZOS), dtype=np.int16)
            th = np.asarray(g.resize((32, 32), Image.BILINEAR), dtype=np.uint8)
            arr = np.asarray(rgb.resize((CACHE_PX, CACHE_PX), Image.BILINEAR), dtype=np.uint8)
    except Exception:
        return None
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return arr, int("".join("1" if b else "0" for b in bits), 2), th.reshape(-1), hashlib.md5(data).hexdigest()

def find_comp(csv_cols):
    for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob(os.environ.get("LOCAL_GLOB", "/nonexistent"), recursive=True):
        head = open(p).readline().strip().split(",")
        if set(csv_cols) <= set(head) and (Path(p).parent / "train_images").exists(): return Path(p).parent
    raise SystemExit("INPUT MISSING: no train.csv with columns %s and a train_images folder. Join the competition and "
                     "add it with Add Input -> Competitions. Inputs present: %s" % (csv_cols, os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "-"))

def list_dataset(name):
    """-> list of (path, label_name, official_split)"""
    import csv
    if name == "plantdoc":
        root = Path(os.environ.get("PLANTDOC_DIR", str(TMP / "PlantDoc-Dataset")))
        if not root.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset.git", str(root)], check=True)
        out = []
        for split in ("train", "test"):
            for c in sorted(d.name for d in (root / split).iterdir() if d.is_dir()):
                for p in sorted((root / split / c).iterdir()):
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}: out.append((str(p), c, split))
        return out
    if name == "cassava2020":
        d = find_comp(["image_id", "label"]); rows = list(csv.DictReader(open(d / "train.csv")))
        names = json.load(open(d / "label_num_to_disease_map.json")) if (d / "label_num_to_disease_map.json").exists() else {}
        return [(str(d / "train_images" / r["image_id"]), names.get(r["label"], r["label"]), "all") for r in rows]
    if name == "paddydoctor":
        d = find_comp(["image_id", "label", "variety"]); rows = list(csv.DictReader(open(d / "train.csv")))
        return [(str(d / "train_images" / r["label"] / r["image_id"]), r["label"], "all") for r in rows]
    if name == "plantwild":                                     # PlantWild v1 (89 classes), Hugging Face mirror
        root = Path(os.environ.get("PLANTWILD_DIR", str(TMP / "plantwild" / "images")))
        if not root.exists():
            from huggingface_hub import hf_hub_download
            import zipfile
            z = hf_hub_download("uqtwei2/PlantWild", "plantwild.zip", repo_type="dataset", local_dir=str(TMP))
            zipfile.ZipFile(z).extractall(TMP); os.remove(z)
        return [(str(p), c.name, "all") for c in sorted(root.iterdir()) if c.is_dir()
                for p in sorted(c.iterdir()) if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    if name == "rice_leaf":                                     # Rice Leaf Disease (6 classes), AgML parquet mirror
        import pyarrow.parquet as pq
        root = Path(os.environ.get("RICE_DIR", str(TMP / "rice_leaf")))
        if not list(root.rglob("*.parquet")):
            from huggingface_hub import snapshot_download
            snapshot_download("Project-AgML/rice_leaf_disease_classification", repo_type="dataset", local_dir=str(root), max_workers=2)
        out = []
        for f in sorted(root.rglob("*.parquet")):
            t = pq.read_table(f).to_pydict()
            out += [(im["bytes"], "class_%s" % lab, "all") for im, lab in zip(t["image"], t["label"])]
        return out
    raise KeyError(name)

def load_dataset(name):
    items = list_dataset(name)
    if SMOKE: items = items[::25]
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=8) as ex: dec = list(ex.map(decode, [p for p, _, _ in items]))
    keep = [i for i, d in enumerate(dec) if d is not None]
    X = np.stack([dec[i][0] for i in keep]); H = np.array([dec[i][1] for i in keep], dtype=np.uint64)
    TH = np.stack([dec[i][2] for i in keep]); MD5 = [dec[i][3] for i in keep]
    labels = [items[i][1] for i in keep]; split = np.array([items[i][2] for i in keep])
    paths = [items[i][0] if isinstance(items[i][0], str) else "%s#%d" % (name, i) for i in keep]
    classes = sorted(set(labels)); y = np.array([classes.index(l) for l in labels])
    log("%s: %d images, %d classes, %d unreadable" % (name, len(keep), len(classes), len(items) - len(keep)))
    return dict(X=X, H=H, TH=TH, MD5=MD5, y=y, split=split, classes=classes, paths=paths, unreadable=len(items) - len(keep))

In [ ]:
# audit: exact retrieval within the radius (multi-index hashing), then thumbnail verification
def candidate_pairs(h, r=RADIUS):
    nb = r + 1; widths = [64 // nb + (1 if k < 64 % nb else 0) for k in range(nb)]
    found, shift = set(), 0
    for w in widths:
        band = (h >> np.uint64(shift)) & np.uint64((1 << w) - 1); shift += w
        order = np.argsort(band, kind="stable"); b = band[order]
        for grp in np.split(order, np.flatnonzero(np.diff(b)) + 1):
            if len(grp) < 2: continue
            g = np.sort(grp[:4000]); x = h[g][:, None] ^ h[g][None, :]
            d = np.unpackbits(x.view(np.uint8).reshape(len(g), len(g), 8), axis=2).sum(2)
            ii, jj = np.where(np.triu(d <= r, 1)); found.update(zip(g[ii].tolist(), g[jj].tolist()))
    P = np.array(sorted(found), dtype=np.int64).reshape(-1, 2)
    dist = np.unpackbits((h[P[:, 0]] ^ h[P[:, 1]]).view(np.uint8).reshape(-1, 8), axis=1).sum(1) if len(P) else np.zeros(0, int)
    return P, dist
def thumb_corr(T, P):
    a = T[P[:, 0]].astype(np.float32); b = T[P[:, 1]].astype(np.float32)
    a -= a.mean(1, keepdims=True); b -= b.mean(1, keepdims=True)
    den = np.sqrt((a * a).sum(1) * (b * b).sum(1)); den[den == 0] = 1
    return (a * b).sum(1) / den
def components(n, P):
    par = list(range(n))
    def f(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for i, j in P: par[f(i)] = f(j)
    g = defaultdict(list)
    for i in range(n): g[f(i)].append(i)
    return [v for v in g.values() if len(v) > 1]

def audit(D):
    n = len(D["y"]); P, dist = candidate_pairs(D["H"]); corr = thumb_corr(D["TH"], P) if len(P) else np.zeros(0)
    V = P[corr >= VERIFY]
    groups = components(n, V)
    conflict = [g for g in groups if len(set(D["y"][g].tolist())) > 1]
    md5 = defaultdict(list)
    for i, m in enumerate(D["MD5"]): md5[m].append(i)
    byte = [g for g in md5.values() if len(g) > 1]
    a = {"images": n, "classes": len(D["classes"]), "unreadable": D["unreadable"],
         "class_counts": {c: int((D["y"] == k).sum()) for k, c in enumerate(D["classes"])},
         "hash_candidates": int(len(P)), "verified_pairs": int(len(V)), "identical_hash_verified_pairs": int(((dist == 0) & (corr >= VERIFY)).sum()),
         "duplicate_groups": len(groups), "images_in_groups": sum(map(len, groups)),
         "pct_images_in_groups": 100 * sum(map(len, groups)) / n, "redundant_copies": sum(map(len, groups)) - len(groups),
         "conflicting_groups": len(conflict), "images_in_conflicting_groups": sum(map(len, conflict)),
         "byte_identical_groups": len(byte), "byte_identical_conflicting": sum(len(set(D["y"][g].tolist())) > 1 for g in byte)}
    cnt = np.bincount(D["y"], minlength=len(D["classes"])); a["imbalance_ratio"] = float(cnt.max() / max(cnt.min(), 1))
    # exposure of a naive random image-level 80/20 split (the protocol used when no official split exists)
    exp = []
    for s in range(10):
        rng = np.random.default_rng(s); te = np.zeros(n, bool)
        for c in range(len(D["classes"])):
            idx = np.flatnonzero(D["y"] == c); rng.shuffle(idx); te[idx[:int(round(0.2 * len(idx)))]] = True
        exp.append(100 * len({(i if te[i] else j) for i, j in V if te[i] != te[j]}) / te.sum())
    a["naive_split_pct_test_with_train_copy"] = {"mean": float(np.mean(exp)), "min": float(np.min(exp)), "max": float(np.max(exp))}
    return a, groups, conflict

In [ ]:
# fixed clean test set and the two training sets
def make_split(D, groups, conflict, name):
    n = len(D["y"]); gid = np.arange(n)
    for k, g in enumerate(groups): gid[g] = n + k                 # one id per duplicate group
    in_conflict = np.zeros(n, bool)
    for g in conflict: in_conflict[g] = True
    if (D["split"] == "test").any():                               # official split (PlantDoc)
        is_te = D["split"] == "test"
        tr_gids = set(gid[~is_te].tolist())
        drop_te = is_te & (np.isin(gid, list(tr_gids)) | in_conflict)   # test copies of train images, conflicting labels
        test = np.flatnonzero(is_te & ~drop_te); train = np.flatnonzero(~is_te)
        note = "official split; %d test images removed (copy of a training image or conflicting label)" % int(drop_te.sum())
    else:                                                           # group-aware stratified 80/20 split, seed 0
        rng = np.random.default_rng(0); is_te = np.zeros(n, bool); members = defaultdict(list)
        for i, g in enumerate(gid.tolist()): members[g].append(i)
        ug = sorted(members)
        g_label = {g: int(np.bincount(D["y"][members[g]]).argmax()) for g in ug}
        g_conf = {g: bool(in_conflict[members[g]].any()) for g in ug}
        for c in range(len(D["classes"])):
            cand = [g for g in ug if g_label[g] == c and not g_conf[g]]; rng.shuffle(cand)
            need = 0.2 * (D["y"] == c).sum(); tot = 0
            for g in cand:
                if tot >= need: break
                is_te[members[g]] = True; tot += len(members[g])
        test = np.flatnonzero(is_te); train = np.flatnonzero(~is_te)
        note = "group-aware stratified 80/20 split (seed 0); conflicting groups kept out of the test set"
    # cleaned training set: drop conflicting groups entirely, keep one image of every other duplicate group
    keep = np.ones(n, bool); keep[in_conflict] = False
    for g in groups:
        if not in_conflict[g[0]]:
            for i in sorted(g)[1:]: keep[i] = False
    train_clean = train[keep[train]]
    s = {"note": note, "test": int(len(test)), "train_raw": int(len(train)), "train_clean": int(len(train_clean)),
         "removed_from_train": int(len(train) - len(train_clean)),
         "test_classes_present": int(len(np.unique(D["y"][test])))}
    log("%s split: %s" % (name, s))
    return test, train, train_clean, s

In [ ]:
# training
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.3, 0.3, 0.3, 0.05), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
                               transforms.RandomErasing(p=0.25)])
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
class ArrDS(Dataset):
    def __init__(self, X, y, idx, tf): self.X, self.y, self.idx, self.tf = X, y, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]; return self.tf(Image.fromarray(self.X[i])), int(self.y[i])

def focal_loss(logits, y, gamma=2.0):
    logp = F.log_softmax(logits.float(), 1).gather(1, y[:, None]).squeeze(1)
    return (-(1 - logp.exp()) ** gamma * logp).mean()

def train_eval(D, tr, te, arch, balance, seed):
    seed_all(seed); nc = len(D["classes"]); ytr = D["y"][tr]
    cnt = np.bincount(ytr, minlength=nc).astype(np.float64)
    w = np.where(cnt > 0, len(ytr) / (nc * np.maximum(cnt, 1)), 0.0)          # inverse-frequency class weights
    if balance == "oversample":
        sampler = WeightedRandomSampler(torch.as_tensor(w[ytr], dtype=torch.double), num_samples=len(tr), replacement=True,
                                        generator=torch.Generator().manual_seed(seed))
        dl = DataLoader(ArrDS(D["X"], D["y"], tr, train_tf), batch_size=BATCH, sampler=sampler, num_workers=WORKERS, pin_memory=True, drop_last=True)
    else:
        dl = DataLoader(ArrDS(D["X"], D["y"], tr, train_tf), batch_size=BATCH, shuffle=True, num_workers=WORKERS, pin_memory=True, drop_last=True,
                        generator=torch.Generator().manual_seed(seed))
    wt = torch.as_tensor(w, dtype=torch.float32, device=DEV)
    model = timm.create_model(arch, pretrained=True, num_classes=nc).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=LR_VIT if "vit" in arch else LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    for ep in range(EPOCHS):
        model.train(); tot = nb = 0
        for x, y in dl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEV == "cuda"):
                out = model(x)
            if balance == "weighted": loss = F.cross_entropy(out.float(), y, weight=wt)
            elif balance == "focal": loss = focal_loss(out, y)
            else: loss = F.cross_entropy(out.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item(); nb += 1
        sched.step()
    model.eval(); P = []
    with torch.no_grad():
        for x, _ in DataLoader(ArrDS(D["X"], D["y"], te, eval_tf), batch_size=128, num_workers=WORKERS):
            with torch.amp.autocast("cuda", enabled=DEV == "cuda"):
                P.append(torch.softmax(model(x.to(DEV)).float(), 1).cpu().numpy())
    del model; torch.cuda.empty_cache()
    return np.concatenate(P), tot / max(nb, 1)

def metrics(y, P, small):
    from sklearn.metrics import f1_score, balanced_accuracy_score
    pred = P.argmax(1); conf = P.max(1); ok = pred == y
    e = 0.0; edges = np.linspace(0, 1, 16)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    rec = {int(c): float(ok[y == c].mean()) for c in np.unique(y)}
    sm = [rec[c] for c in small if c in rec]
    return {"accuracy": 100 * float(ok.mean()), "macro_f1": 100 * float(f1_score(y, pred, average="macro", zero_division=0)),
            "balanced_accuracy": 100 * float(balanced_accuracy_score(y, pred)), "ece": float(e),
            "minority_recall": 100 * float(np.mean(sm)) if sm else None}

In [ ]:
# hybrid fusion: frozen foundation-model features (DINOv2-L, CLIP ViT-L) + fine-tuned EfficientNet-B0 features
FUSION_RUN = ["plantdoc", "rice_leaf", "paddydoctor", "plantwild", "cassava2020"]
if os.environ.get("FUSION_RUN"): FUSION_RUN = os.environ["FUSION_RUN"].split(",")
FROZEN = {"dinov2_l": os.environ.get("DINO_NAME", "vit_large_patch14_dinov2.lvd142m"),
          "clip_l": os.environ.get("CLIP_NAME", "vit_large_patch14_clip_224.openai")}
HEAD_EPOCHS, HEAD_BS, HEAD_LR, HEAD_WD, PROTOS = 60, 256, 1e-3, 1e-4, 4
if SMOKE: HEAD_EPOCHS = 5
RESULTS["_settings"].update({"frozen": FROZEN, "head_epochs": HEAD_EPOCHS, "head_lr": HEAD_LR, "protos_per_class": PROTOS})

class FeatDS(Dataset):
    def __init__(self, X, idx, tf): self.X, self.idx, self.tf = X, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, k): return self.tf(Image.fromarray(self.X[self.idx[k]]))

@torch.no_grad()
def extract(model, X, idx, tf):
    model.eval(); out = []
    for x in DataLoader(FeatDS(X, idx, tf), batch_size=128, num_workers=WORKERS):
        with torch.amp.autocast("cuda", enabled=DEV == "cuda"):
            out.append(model(x.to(DEV)).float().cpu().numpy())
    return np.concatenate(out)

def frozen_features(D, idx):
    """frozen backbone features for every image in idx (one pass per backbone)"""
    feats = {}
    for key, name in FROZEN.items():
        m = timm.create_model(name, pretrained=True, num_classes=0, img_size=IMG).to(DEV)
        cfg = timm.data.resolve_data_config({}, model=m)
        tf = transforms.Compose([transforms.Resize((IMG, IMG), interpolation=transforms.InterpolationMode.BICUBIC),
                                 transforms.ToTensor(), transforms.Normalize(cfg["mean"], cfg["std"])])
        t = time.time(); feats[key] = extract(m, D["X"], idx, tf); log("  %s features %s (%.1f min)" % (key, feats[key].shape, (time.time() - t) / 60))
        del m; torch.cuda.empty_cache()
    return feats

def finetune_cnn(D, tr, te_all, seed):
    """EfficientNet-B0 fine-tuned on the raw training set (same recipe as the grid); returns its test softmax on te_all
    and penultimate features for tr and te_all"""
    seed_all(seed); nc = len(D["classes"])
    dl = DataLoader(ArrDS(D["X"], D["y"], tr, train_tf), batch_size=BATCH, shuffle=True, num_workers=WORKERS, pin_memory=True,
                    drop_last=True, generator=torch.Generator().manual_seed(seed))
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=nc).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=LR); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    for ep in range(EPOCHS):
        model.train()
        for x, y in dl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEV == "cuda"):
                out = model(x)
            loss = F.cross_entropy(out.float(), y); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
    class Pre(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x): return s.m.forward_head(s.m.forward_features(x), pre_logits=True)
    @torch.no_grad()
    def probs(idx):
        model.eval(); P = []
        for x, _ in DataLoader(ArrDS(D["X"], D["y"], idx, eval_tf), batch_size=128, num_workers=WORKERS):
            with torch.amp.autocast("cuda", enabled=DEV == "cuda"):
                P.append(torch.softmax(model(x.to(DEV)).float(), 1).cpu().numpy())
        return np.concatenate(P)
    f_tr = extract(Pre(model), D["X"], tr, eval_tf); f_te = extract(Pre(model), D["X"], te_all, eval_tf); P_te = probs(te_all)
    del model; torch.cuda.empty_cache()
    return P_te, f_tr, f_te

class HybridHead(nn.Module):
    """0.5 x linear logits + 0.5 x prototype logits (scaled cosine similarity to the nearest of K prototypes per class)"""
    def __init__(s, d, nc, protos):
        super().__init__(); s.lin = nn.Linear(d, nc); s.P = nn.Parameter(protos); s.scale = nn.Parameter(torch.tensor(10.0)); s.nc = nc
    def forward(s, x):
        sim = F.normalize(x, dim=1) @ F.normalize(s.P, dim=1).T
        return 0.5 * s.lin(x) + 0.5 * s.scale * sim.view(len(x), s.nc, -1).max(-1).values

def prep(blocks_tr, blocks_te):
    """L2-normalise each backbone block, concatenate, z-score with training statistics"""
    n = lambda a: a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-8)
    Ftr = np.concatenate([n(b) for b in blocks_tr], 1); Fte = np.concatenate([n(b) for b in blocks_te], 1)
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
    return ((Ftr - mu) / sd).astype(np.float32), ((Fte - mu) / sd).astype(np.float32)

def train_head(Ftr, Q, w, Fte, seed):
    seed_all(seed); nc = Q.shape[1]; hard = Q.argmax(1); rng = np.random.default_rng(seed)
    protos = []
    for c in range(nc):
        idx = np.flatnonzero(hard == c)
        pick = rng.choice(idx, PROTOS, replace=len(idx) < PROTOS) if len(idx) else rng.choice(len(Ftr), PROTOS)
        protos.append(Ftr[pick])
    head = HybridHead(Ftr.shape[1], nc, torch.tensor(np.concatenate(protos))).to(DEV)
    opt = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=HEAD_WD)
    steps = HEAD_EPOCHS * int(np.ceil(len(Ftr) / HEAD_BS)); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    Xt, Qt, Wt = torch.tensor(Ftr, device=DEV), torch.tensor(Q, device=DEV), torch.tensor(w, device=DEV)
    g = torch.Generator(device="cpu").manual_seed(seed)
    for ep in range(HEAD_EPOCHS):
        perm = torch.randperm(len(Ftr), generator=g).to(DEV)
        for k in range(0, len(Ftr), HEAD_BS):
            b = perm[k:k + HEAD_BS]; logp = F.log_softmax(head(Xt[b]), 1)
            loss = (-(Qt[b] * logp).sum(1) * Wt[b]).sum() / Wt[b].sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    head.eval()
    with torch.no_grad(): return torch.softmax(head(torch.tensor(Fte, device=DEV)), 1).cpu().numpy()

def label_variants(D, train, train_clean, groups, conflict):
    """keep: raw training set, hard labels. drop: cleaned training set (conflicting groups removed, one image per
    duplicate group). soft: raw training set; members of a conflicting group get the group's label distribution
    (training members only) and every duplicate group carries a total weight of one."""
    nc = len(D["classes"]); pos = {int(i): k for k, i in enumerate(train)}
    Q = np.eye(nc, dtype=np.float32)[D["y"][train]]; w = np.ones(len(train), np.float32)
    n_soft = 0
    for g in groups:
        mem = [pos[i] for i in g if i in pos]
        if len(mem) < 2: continue
        w[mem] = 1.0 / len(mem)
        labs = D["y"][train][mem]
        if len(set(labs.tolist())) > 1:
            Q[mem] = np.bincount(labs, minlength=nc) / len(labs); n_soft += len(mem)
    keep_pos = np.array([pos[int(i)] for i in train_clean])
    return {"keep": (np.arange(len(train)), np.eye(nc, dtype=np.float32)[D["y"][train]], np.ones(len(train), np.float32)),
            "drop": (keep_pos, np.eye(nc, dtype=np.float32)[D["y"][train_clean]], np.ones(len(keep_pos), np.float32)),
            "soft": (np.arange(len(train)), Q, w)}, n_soft

In [ ]:
# integrity-aware adaptive fusion: validation-gated combination of a fine-tuned CNN branch and a frozen
# foundation-model branch, conflict-aware soft labels, temperature scaling. Every choice uses a held-out validation split.
ADA_RUN = ["plantdoc", "rice_leaf", "paddydoctor", "plantwild", "cassava2020"]
if os.environ.get("ADA_RUN"): ADA_RUN = os.environ["ADA_RUN"].split(",")
VAL_FRAC, VAL_SEED = 0.10, 1
ALPHAS = np.round(np.linspace(0, 1, 21), 2)
TEMPS = np.exp(np.linspace(np.log(0.05), np.log(20), 241))
RESULTS["_settings"].update({"val_frac": VAL_FRAC, "val_seed": VAL_SEED, "alpha_grid": ALPHAS.tolist()})

def make_val(D, train, groups, conflict):
    """group-aware stratified 10% of the training set; conflicting groups never enter validation; classes with fewer
    than 5 training images give no validation images"""
    n = len(D["y"]); gid = np.arange(n)
    for k, g in enumerate(groups): gid[g] = n + k
    bad = np.zeros(n, bool)
    for g in conflict: bad[g] = True
    members = defaultdict(list)
    for i in train: members[int(gid[i])].append(int(i))
    rng = np.random.default_rng(VAL_SEED); val = []
    for c in range(len(D["classes"])):
        cand = [g for g, m in members.items() if np.bincount(D["y"][m]).argmax() == c and not bad[m].any()]
        need = int(np.floor(VAL_FRAC * sum(len(members[g]) for g in members if np.bincount(D["y"][members[g]]).argmax() == c)))
        if (D["y"][train] == c).sum() < 5: continue
        rng.shuffle(cand); tot = 0
        for g in cand:
            if tot >= need: break
            val += members[g]; tot += len(members[g])
    val = np.array(sorted(val), dtype=np.int64)
    return val, np.setdiff1d(train, val)

def logp(P): return np.log(np.clip(P, 1e-8, 1.0))
def nll(L, y, T=1.0):
    Z = L / T; Z = Z - Z.max(1, keepdims=True); lp = Z - np.log(np.exp(Z).sum(1, keepdims=True))
    return float(-lp[np.arange(len(y)), y].mean())
def soft(L, T=1.0):
    Z = L / T; Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)
def fit_temp(L, y): return float(TEMPS[np.argmin([nll(L, y, t) for t in TEMPS])]) if len(y) else 1.0   # no validation data: no scaling

def fit_gate(Lc, Lf, y):
    """alpha weights the CNN branch: L = alpha * log p_cnn + (1 - alpha) * log p_foundation; alpha and the temperature
    minimise validation NLL"""
    if not len(y): return 0.5, 1.0, float("nan")                 # no validation data: equal weights
    best = None
    for a in ALPHAS:
        L = a * Lc + (1 - a) * Lf; T = fit_temp(L, y); v = nll(L, y, T)
        if best is None or v < best[0] - 1e-9: best = (v, float(a), T)
    return best[1], best[2], best[0]

In [ ]:
# adaptive fusion main loop
for name in ADA_RUN:
    log("=" * 20 + " adaptive " + name)
    D = load_dataset(name)
    a_, groups, conflict = audit(D)
    test, train, train_clean, s = make_split(D, groups, conflict, name)
    val, tr90 = make_val(D, train, groups, conflict)
    tr90_clean = np.intersect1d(train_clean, tr90)
    ev = np.concatenate([val, test]); nv = len(val)
    yv, yt = D["y"][val], D["y"][test]
    np.savez_compressed(OUTD / ("%s_ada_split.npz" % name), test=test, train=train, val=val, tr90=tr90, tr90_clean=tr90_clean)
    variants, n_soft = label_variants(D, tr90, tr90_clean, groups, conflict)
    cnt = np.bincount(D["y"][train], minlength=len(D["classes"])); small = [int(c) for c in np.argsort(cnt)[: max(1, len(cnt) // 3)]]
    R = RESULTS.setdefault("adaptive", {}).setdefault(name, {"split": s, "val": int(nv), "train90": int(len(tr90)), "n_soft": n_soft, "runs": [], "gate": []})
    log("%s: val %d, train90 %d, soft-label images %d" % (name, nv, len(tr90), n_soft))
    fz = frozen_features(D, np.concatenate([tr90, ev])); n90 = len(tr90)
    fz_tr = {k: v[:n90] for k, v in fz.items()}; fz_ev = {k: v[n90:] for k, v in fz.items()}
    def record(model, labels, seed, P_ev, calib):
        """P_ev: probabilities on [val, test]; calib: raw metrics and metrics after temperature scaling fitted on val"""
        np.save(OUTD / ("%s_ada_%s_%s_%d.npy" % (name, model, labels, seed)), P_ev.astype(np.float16))
        L = logp(P_ev); T = fit_temp(L[:nv], yv)
        for kind, P in (("raw", P_ev[nv:]), ("ts", soft(L[nv:], T))):
            m = metrics(yt, P, small); m.update({"model": model, "labels": labels, "seed": seed, "calib": kind, "T": T}); R["runs"].append(m)
        log("%s %-14s %-5s seed %d: acc %.2f macroF1 %.2f ece %.3f (ts %.3f)" % (name, model, labels, seed, R["runs"][-2]["accuracy"],
            R["runs"][-2]["macro_f1"], R["runs"][-2]["ece"], R["runs"][-1]["ece"]))
    for seed in SEEDS:
        t = time.time(); P_cnn, c_tr, c_ev = finetune_cnn(D, tr90, ev, seed); log("  cnn fine-tune seed %d (%.1f min)" % (seed, (time.time() - t) / 60))
        record("effnet_ft", "keep", seed, P_cnn, True)
        blocks = {"dinov2_l": (fz_tr["dinov2_l"], fz_ev["dinov2_l"]), "clip_l": (fz_tr["clip_l"], fz_ev["clip_l"]), "effnet": (c_tr, c_ev)}
        combos = {"dinov2_l": ["dinov2_l"], "clip_l": ["clip_l"], "fusion2": ["dinov2_l", "clip_l"], "fusion3": ["effnet", "dinov2_l", "clip_l"]}
        P_f2 = {}
        for model, parts in combos.items():
            Ftr, Fev = prep([blocks[p][0] for p in parts], [blocks[p][1] for p in parts])
            for labels in ("keep", "soft"):
                if model in ("dinov2_l", "clip_l") and labels != "keep": continue
                rows, Q, w = variants[labels]; P = train_head(Ftr[rows], Q, w, Fev, seed); record(model, labels, seed, P, True)
                if model == "fusion2": P_f2[labels] = P
        for labels in ("keep", "soft"):                       # the proposed model uses soft; keep is the ablation
            Lc, Lf = logp(P_cnn), logp(P_f2[labels])
            alpha, T, v = fit_gate(Lc[:nv], Lf[:nv], yv)
            P_ada = soft(alpha * Lc + (1 - alpha) * Lf, 1.0)          # temperature applied inside record (fitted on val)
            R["gate"].append({"labels": labels, "seed": seed, "alpha": alpha, "T_joint": T, "val_nll": v})
            log("  gate %s seed %d: alpha %.2f (weight of the CNN branch), T %.2f" % (labels, seed, alpha, T))
            record("adaptive", labels, seed, P_ada, True)
        save()
    np.save(OUTD / ("%s_ada_eval_labels.npy" % name), D["y"][ev])
    del D, fz, fz_tr, fz_ev; import gc; gc.collect()

In [ ]:
# print everything (survives a failed download)
save(); log("finished")
print("=" * 30, "COPY EVERYTHING BELOW", "=" * 30)
print(json.dumps({k: {"gate": v["gate"], "split": v["split"], "val": v["val"]} for k, v in RESULTS.get("adaptive", {}).items()}, indent=1))